# Phase 1 Cross-LLM Baseline — Claude 1:1 Matching

Run 1:1 schema matching with Claude on all 9 ground-truth relation pairs and
evaluate against `updated_ground_truth.csv`.

In [8]:
# Cell 1 — environment setup (MUST run before any thesis-extension import)
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

# Detect thesis-extension root regardless of CWD
_cwd = Path(os.getcwd()).resolve()
for _candidate in [_cwd, _cwd.parent, _cwd / "thesis-extension"]:
    if (_candidate / "pipeline.py").exists():
        _thesis_root = _candidate
        break
else:
    raise RuntimeError(
        f"Cannot locate thesis-extension root from CWD={_cwd}. "
        "Run this notebook from the thesis-extension/ directory."
    )

_notebooks_dir = _thesis_root / "notebooks"
_project_root = _thesis_root.parent

# Set path-based env defaults (must happen before importing config)
os.environ.setdefault("RESULTS_DIR", str(_thesis_root / "results"))
os.environ.setdefault("TEMPLATE_DIR", str(_thesis_root / "templates"))
os.environ.setdefault("LLM_PROVIDER", "anthropic")

# Load .env (does not override shell-set vars)
load_dotenv(_thesis_root / ".env", override=False)

# Add thesis-extension and notebooks to sys.path
for _d in [str(_thesis_root), str(_notebooks_dir)]:
    if _d not in sys.path:
        sys.path.insert(0, _d)

print("thesis-extension root:", _thesis_root)
print("RESULTS_DIR          :", os.environ["RESULTS_DIR"])
print("TEMPLATE_DIR         :", os.environ["TEMPLATE_DIR"])

thesis-extension root: /Users/sadiazaman/Documents/my-improvement-project/thesis-extension
RESULTS_DIR          : /Users/sadiazaman/Documents/my-improvement-project/thesis-extension/results
TEMPLATE_DIR         : /Users/sadiazaman/Documents/my-improvement-project/thesis-extension/templates


In [9]:
# Cell 2 — imports and mock-mode check
from config import config
from models import Feedback, Parameters, Side
from pipeline import schema_match
from evaluation import evaluate_against_ground_truth
from utils import RELATION_PAIRS, load_relation

if not config["QUERY_LLM"]:
    print("WARNING: QUERY_LLM=False — running in mock mode. LLM will NOT be called.")

GT_PATH = str(_project_root / "updated_ground_truth.csv")
_model = (
    config["ANTHROPIC_MODEL"]
    if config["LLM_PROVIDER"] == "anthropic"
    else config["OPENAI_MODEL"]
)

print(f"Provider : {config['LLM_PROVIDER']}")
print(f"Model    : {_model}")
print(f"GT path  : {GT_PATH}")

Provider : anthropic
Model    : claude-sonnet-4-20250514
GT path  : /Users/sadiazaman/Documents/my-improvement-project/updated_ground_truth.csv


In [10]:
# Cell 3 — load all 9 relation pairs
LOADED_PAIRS = []
for src_name, src_schema, tgt_name, tgt_schema in RELATION_PAIRS:
    src = load_relation(src_name, src_schema, Side.SOURCE)
    tgt = load_relation(tgt_name, tgt_schema, Side.TARGET)
    LOADED_PAIRS.append((src, tgt))
    print(
        f"  {src_name:20s} ({len(src.attributes):2d} attrs)"
        f" -> {tgt_name:22s} ({len(tgt.attributes):2d} attrs)"
    )
print(f"\nLoaded {len(LOADED_PAIRS)} relation pairs.")

  Patients             ( 6 attrs) -> Person                 (18 attrs)
  Admissions           (16 attrs) -> Visit_Occurrence       (17 attrs)
  Admissions           (16 attrs) -> Death                  ( 7 attrs)
  Prescriptions        (17 attrs) -> Drug_Exposure          (23 attrs)
  Diagnoses_ICD        ( 5 attrs) -> Condition_Occurrence   (16 attrs)
  Transfers            ( 7 attrs) -> Care_Site              ( 6 attrs)
  Transfers            ( 7 attrs) -> Visit_Detail           (19 attrs)
  Admissions           (16 attrs) -> Visit_Detail           (19 attrs)
  Services             ( 5 attrs) -> Visit_Detail           (19 attrs)

Loaded 9 relation pairs.


In [12]:
import nest_asyncio
nest_asyncio.apply()

In [13]:
# Cell 4 — run schema_match for each pair
results = []
for src, tgt in LOADED_PAIRS:
    params = Parameters(
        source_relation=src,
        target_relation=tgt,
        llm_model=_model,
    )
    result = schema_match(params)
    results.append(result)
    print(
        f"  {src.name}->{tgt.name}: "
        f"{len(result.pairs)} 1:1 pairs, "
        f"{len(result.group_pairs)} group pairs"
    )

  Patients->Person: 108 1:1 pairs, 1 group pairs


KeyboardInterrupt: 

In [5]:
# Cell 5 — evaluate and build summary DataFrame
import pandas as pd

rows = []
for result in results:
    report = evaluate_against_ground_truth(result, GT_PATH)
    rows.append({
        "pair": (
            f"{result.parameters.source_relation.name}"
            f"->{result.parameters.target_relation.name}"
        ),
        "precision_1to1": round(report.precision_1to1, 3),
        "recall_1to1":    round(report.recall_1to1, 3),
        "f1_1to1":        round(report.f1_1to1, 3),
        "precision_group": round(report.precision_group, 3),
        "recall_group":    round(report.recall_group, 3),
        "f1_group":        round(report.f1_group, 3),
    })

summary_df = pd.DataFrame(rows)
print(summary_df.to_string(index=False))
print(f"\nMean F1 (1:1)  : {summary_df['f1_1to1'].mean():.3f}")
print(f"Mean F1 (group): {summary_df['f1_group'].mean():.3f}")

                               pair  precision_1to1  recall_1to1  f1_1to1  precision_group  recall_group  f1_group
                   Patients->Person             1.0        0.057    0.108              1.0           0.5     0.667
       Admissions->Visit_Occurrence             0.0        0.000    0.000              0.0           0.0     0.000
                  Admissions->Death             0.0        0.000    0.000              0.0           0.0     0.000
       Prescriptions->Drug_Exposure             0.0        0.000    0.000              0.0           0.0     0.000
Diagnoses_ICD->Condition_Occurrence             0.0        0.000    0.000              0.0           0.0     0.000
               Transfers->Care_Site             0.0        0.000    0.000              0.0           0.0     0.000
            Transfers->Visit_Detail             0.0        0.000    0.000              0.0           0.0     0.000
           Admissions->Visit_Detail             0.0        0.000    0.000       

In [6]:
# Cell 6 — save summary CSV
out_dir = Path(os.environ["RESULTS_DIR"])
out_dir.mkdir(parents=True, exist_ok=True)
csv_path = out_dir / "claude_baseline_results.csv"
summary_df.to_csv(csv_path, index=False)
print(f"Saved: {csv_path}")

Saved: /Users/sadiazaman/Documents/my-improvement-project/thesis-extension/results/claude_baseline_results.csv


In [ ]:
# Cell 7 — cost summary from cost_log.jsonl
import json

cost_log = out_dir / "cost_log.jsonl"
if cost_log.exists():
    records = [
        json.loads(line)
        for line in cost_log.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    total_cost   = sum(r.get("cost_usd", 0.0) for r in records)
    total_input  = sum(r.get("input_tokens", 0) for r in records)
    total_output = sum(r.get("output_tokens", 0) for r in records)
    print(f"Total API calls     : {len(records)}")
    print(f"Total input tokens  : {total_input:,}")
    print(f"Total output tokens : {total_output:,}")
    print(f"Estimated cost (USD): ${total_cost:.4f}")
else:
    print("No cost_log.jsonl found (mock mode or first run).")

Total API calls     : 814
Total input tokens  : 0
Total output tokens : 0
Estimated cost (USD): $0.0000
